In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  
import time

from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm

In [ ]:
class StockDataset_by_year(Dataset):
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task
        self.data = []
        self.labels = []
        
        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")
            
            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            
            # 删除 'permno' 和 'DATE' 列，只保留剩余列作为特征
            X = X.drop(columns=['permno', 'DATE'])
            
            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y = (y > 0).astype(int)
            
            # 转换为numpy数组并存储
            self.data.append(X.values)
            self.labels.append(y.values)
        
        # 将数据和标签转换为单个数组
        self.data = torch.tensor(np.vstack(self.data), dtype=torch.float32)
        self.labels = torch.tensor(np.vstack(self.labels), dtype=torch.float32)
        
        # 打印数据集信息
        print(f"Data loaded from {start_year} to {end_year}.")
        print(f"Total samples: {len(self.labels)}")
        print(f"Feature dimension: {self.data.shape[1]}")
        print(f"Task: {self.task}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    


# 构建数据集和数据加载器的函数
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=32):
    # 构建训练集
    print("Constructing training dataset...")
    train_dataset = StockDataset_by_year(base_dir, start_year_train, start_year_test - 1, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 构建测试集
    print("Constructing testing dataset...")
    test_dataset = StockDataset_by_year(base_dir, start_year_test, end_year_test, task)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
import gc
import shutil
import time



# 检查 GPU 可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def calculate_r2_oos(real, predict):
    real = np.array(real)
    predict = np.array(predict)
    assert real.shape == predict.shape 
    valid_mask = ~np.isnan(real)
    numerator = np.sum((real[valid_mask] - predict[valid_mask]) ** 2)
    denominator = np.sum(real[valid_mask] ** 2)
    R2_oos = 1 - (numerator / denominator)
    return R2_oos

In [ ]:
from tqdm import tqdm

import time

from tqdm import tqdm
import time
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

from tqdm import tqdm
import time
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

from tqdm import tqdm
import time
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

from tqdm import tqdm
import time
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

def train_model_once(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10, save_base_path=None, use_scheduler=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Optionally add a learning rate scheduler
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {'train_loss': [], 'test_loss': [], 'test_r2': [], 'learning_rate': []}

    # 创建模型文件夹
    model_folder = os.path.join(save_base_path, model.__class__.__name__)
    if not os.path.exists(model_folder):
        os.makedirs(model_folder)

    total_start_time = time.time()

    # 添加进度条来跟踪epoch进度
    epoch_iterator = tqdm(range(num_epochs), desc='Training Epochs', unit='epoch')
    for epoch in epoch_iterator:
        model.train()
        train_loss = 0.0

        # 在训练数据加载器上添加进度条
        train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False, unit='batch')
        for X_batch, y_batch in train_loader_iter:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)
        results['train_loss'].append(train_loss)

        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        # 在测试数据加载器上添加进度条
        test_loader_iter = tqdm(test_loader, desc=f"Epoch {epoch+1} Testing", leave=False, unit='batch')
        with torch.no_grad():
            for X_batch, y_batch in test_loader_iter:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)
        
        results['test_loss'].append(test_loss)
        results['test_r2'].append(test_r2)

        # Log the current learning rate if scheduler is used
        if scheduler:
            current_lr = scheduler.optimizer.param_groups[0]['lr']
            results['learning_rate'].append(current_lr)

        # 更新进度条的描述信息
        epoch_iterator.set_postfix({
            'Train Loss': f"{train_loss:.6f}", 
            'Test Loss': f"{test_loss:.6f}", 
            'Test R²': f"{test_r2:.6f}",
            'LR': current_lr if use_scheduler else lr
        })

        # Use scheduler if applicable
        if scheduler:
            scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # 保存最佳模型
    if best_model_state:
        model.load_state_dict(best_model_state)
        model_save_path = os.path.join(model_folder, f"one_loop_best_model.pth")
        torch.save(model.state_dict(), model_save_path)

        # 保存训练历史
        history_save_path = os.path.join(model_folder, "one_loop_training_history.txt")
        with open(history_save_path, 'w') as f:
            f.write("Epoch\tTrain Loss\tTest Loss\tTest R²\tLR\n")
            for epoch in range(len(results['train_loss'])):
                f.write(f"{epoch+1}\t{results['train_loss'][epoch]:.6f}\t{results['test_loss'][epoch]:.6f}\t{results['test_r2'][epoch]:.6f}\t{results['learning_rate'][epoch] if use_scheduler else lr:.6f}\n")

        return model_save_path, results['test_r2'][-1], total_training_time
    else:
        return None, results['test_r2'][-1], total_training_time

In [ ]:
# 使用函数创建数据加载器
data_dir = 'data/processed/yearly'
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

train_loader_reg_one_loop, test_loader_reg_one_loop = create_dataloaders(data_dir, start_year_train, start_year_test, end_year_test, task='regression')

In [ ]:
input_dim = train_loader_reg_one_loop.dataset[0][0].shape[0]
base_save_path = 'outputs/models'
print(input_dim)

## NN1

In [ ]:
import torch
import torch.nn as nn

# 定义 NN1 模型
class NN1(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128):  # 添加 hidden_neurons 参数
        super(NN1, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)  # 使用 hidden_neurons 作为隐藏层大小
        self.out = nn.Linear(hidden_neurons, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.out(x)


model = NN1(input_dim, hidden_neurons=128)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.001, patience=10, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## NN2

In [ ]:
import torch
import torch.nn as nn

# 定义 NN2 模型
class NN2(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128, dropout_rate=0.3):
        super(NN2, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)
        self.bn1 = nn.BatchNorm1d(hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons // 2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons // 2)
        self.out = nn.Linear(hidden_neurons // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.out(x)



model = NN2(input_dim, hidden_neurons=128)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=25, lr=0.03, patience=8, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## NN3

In [ ]:
# 定义 NN3 模型
class NN3(nn.Module):
    def __init__(self, input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5):
        super(NN3, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons1)
        self.bn1 = nn.BatchNorm1d(hidden_neurons1)
        self.fc2 = nn.Linear(hidden_neurons1, hidden_neurons2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons2)
        self.fc3 = nn.Linear(hidden_neurons2, hidden_neurons3)
        self.bn3 = nn.BatchNorm1d(hidden_neurons3)
        self.out = nn.Linear(hidden_neurons3, 1)
        self.dropout1 = nn.Dropout(dropout_rate1)
        self.dropout2 = nn.Dropout(dropout_rate2)
        self.dropout3 = nn.Dropout(dropout_rate3)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.out(x)


# 创建 NN3 模型实例

nn3_model = NN3(input_dim=input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    nn3_model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=25, lr=0.03, patience=8, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## 1D CNN

In [ ]:
import torch
import torch.nn as nn

# 定义 1D CNN 模型
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes=1, dropout_rate=0.3):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(128 * (input_dim // 8), 64)  # 计算池化后的特征维度
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # 添加通道维度，转换为 [batch_size, 1, input_dim]
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = x.view(x.size(0), -1)  # 展平特征图
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)


# 创建 1D CNN 模型实例

model = CNN1D(input_dim=input_dim, dropout_rate=0.4)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=25, lr=0.01, patience=8, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## rnn

In [ ]:
# 定义标准 RNN 模型
class StandardRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(StandardRNNModel, self).__init__()
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # 改变输入的形状为 (batch_size, seq_length=1, input_dim)
        x = x.unsqueeze(1)  # 添加一个时间维度 seq_length=1
        x, _ = self.rnn(x)  # 输出 x 的形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)


# 创建标准 RNN 模型实例

model = StandardRNNModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)



# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.03, patience=16, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate)
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = x.unsqueeze(1)  # 添加一个时间维度 seq_length=1
        x, _ = self.lstm(x)  # x 形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)

    
# 创建 RNN 模型实例

lstm_model = LSTMModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    lstm_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.03, patience=16, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## Transformer

In [ ]:
import torch
import torch.nn as nn

# 定义简化版的 Transformer 模型
class SimpleTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)  # 输入嵌入层，将输入维度映射到 d_model
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # x 的形状为 (batch_size, input_dim) 或 (batch_size, seq_length, input_dim)
        if x.dim() == 2:
            x = x.unsqueeze(1)  # 增加一个时间维度，使得 x 的形状为 (batch_size, seq_length=1, input_dim)
            
        x = self.embedding(x)  # 将输入映射到 d_model 维度
        x = x.permute(1, 0, 2)  # 转换为 (seq_length, batch_size, d_model) 供 Transformer 使用
        x = self.transformer_encoder(x)
        x = x.permute(1, 0, 2)  # 转换回 (batch_size, seq_length, d_model)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)

# 创建 Transformer 模型实例

transformer_model = SimpleTransformer(input_dim=input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1)



# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    transformer_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.005, patience=20, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
import numpy as np
import os
import time
from tqdm import tqdm
import torch.optim as optim
import gc

# 定义简化版的 GNN 模型
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super(SimpleGNN, self).__init__()
        
        # 定义图卷积层
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))  # 第一个卷积层，将 input_dim 映射到 hidden_dim
        
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))  # 中间层
        
        self.fc1 = nn.Linear(hidden_dim, 64)  # 全连接层
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)  # 输出层

    def forward(self, x, edge_index, batch):
        # x 的形状为 (num_nodes, input_dim)
        # edge_index 表示边的连接关系，batch 表示节点所属的图
        
        # 依次通过图卷积层
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = self.dropout(x)
        
        # 对节点进行全局池化，得到每个图的图级表示
        x = global_mean_pool(x, batch)  # (batch_size, hidden_dim)
        
        # 全连接层
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        # 输出层
        return self.out(x)

# 创建 GNN 模型实例
#input_dim = 87  # 假设输入维度为 87


# 特供 GNN 的训练方法，改造输入数据为图数据

def transform_to_graph_data(X_batch, y_batch):
    batch_size = X_batch.size(0)
    num_nodes = X_batch.size(0)  # 每个样本被视为一个节点
    edge_index = torch.combinations(torch.arange(num_nodes), r=2).t().contiguous()  # 全连接图
    batch = torch.arange(batch_size, dtype=torch.long).repeat_interleave(1)  # 每个节点属于不同的图
    data = Data(x=X_batch, edge_index=edge_index, y=y_batch, batch=batch)
    return data


# 一次性训练模型并验证
def train_model_once_gnn(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10, save_base_path=None, use_scheduler=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Optionally add a learning rate scheduler
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {'train_loss': [], 'test_loss': [], 'test_r2': [], 'learning_rate': []}

    # 创建模型文件夹
    model_folder = os.path.join(save_base_path, model.__class__.__name__)
    if not os.path.exists(model_folder):
        os.makedirs(model_folder)

    total_start_time = time.time()

    # 添加进度条来跟踪epoch进度
    epoch_iterator = tqdm(range(num_epochs), desc='Training Epochs', unit='epoch')
    for epoch in epoch_iterator:
        model.train()
        train_loss = 0.0

        # 在训练数据加载器上添加进度条
        train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False, unit='batch')
        for X_batch, y_batch in train_loader_iter:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            # 转换为图数据
            graph_data = transform_to_graph_data(X_batch, y_batch).to(device)
            
            # 前向传播
            outputs = model(graph_data.x, graph_data.edge_index, graph_data.batch)
            loss = criterion(outputs, graph_data.y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)
        results['train_loss'].append(train_loss)

        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        # 在测试数据加载器上添加进度条
        test_loader_iter = tqdm(test_loader, desc=f"Epoch {epoch+1} Testing", leave=False, unit='batch')
        with torch.no_grad():
            for X_batch, y_batch in test_loader_iter:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                # 转换为图数据
                graph_data = transform_to_graph_data(X_batch, y_batch).to(device)
                
                # 前向传播
                outputs = model(graph_data.x, graph_data.edge_index, graph_data.batch)
                loss = criterion(outputs, graph_data.y)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)
        
        results['test_loss'].append(test_loss)
        results['test_r2'].append(test_r2)

        # Log the current learning rate if scheduler is used
        if scheduler:
            current_lr = scheduler.optimizer.param_groups[0]['lr']
            results['learning_rate'].append(current_lr)

        # 更新进度条的描述信息
        epoch_iterator.set_postfix({
            'Train Loss': f"{train_loss:.6f}", 
            'Test Loss': f"{test_loss:.6f}", 
            'Test R²': f"{test_r2:.6f}",
            'LR': current_lr if use_scheduler else lr
        })

        # Use scheduler if applicable
        if scheduler:
            scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # 保存最佳模型
    if best_model_state:
        model.load_state_dict(best_model_state)
        model_save_path = os.path.join(model_folder, f"one_loop_best_model.pth")
        torch.save(model.state_dict(), model_save_path)

        # 保存训练历史
        history_save_path = os.path.join(model_folder, "one_loop_training_history.txt")
        with open(history_save_path, 'w') as f:
            f.write("Epoch\tTrain Loss\tTest Loss\tTest R²\tLR\n")
            for epoch in range(len(results['train_loss'])):
                f.write(f"{epoch+1}\t{results['train_loss'][epoch]:.6f}\t{results['test_loss'][epoch]:.6f}\t{results['test_r2'][epoch]:.6f}\t{results['learning_rate'][epoch] if use_scheduler else lr:.6f}\n")

        return model_save_path, results['test_r2'][-1], total_training_time
    else:
        return None, results['test_r2'][-1], total_training_time
    




gnn_model = SimpleGNN(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout=0.3)


# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once_gnn(
    gnn_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.05, patience=30, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()